In [2]:
import requests
import os
import shutil

In [3]:
years = list(range(1991,2020))

In [4]:
url_start = "https://www.basketball-reference.com/awards/awards_{}.html"

In [ ]:
for year in years:
    url = url_start.format(year)
    data = requests.get(url)

    with open("mvp/{}.html".format(year), "w+") as f:
        f.write(data.text)

In [5]:
from bs4 import BeautifulSoup

In [6]:
with open("mvp/1991.html") as f:
    page = f.read()
    
soup = BeautifulSoup(page, 'html.parser')
soup.find('tr', class_="over_header").decompose()

In [7]:
mvp_table = soup.find(id="mvp")

In [8]:
import pandas as pd
from io import StringIO

In [9]:
mvp_1991 = pd.read_html(StringIO(str(mvp_table)))[0]

In [10]:
dfs = []
for year in years:
    with open("mvp/{}.html".format(year)) as f:
        page = f.read()
    
    soup = BeautifulSoup(page, 'html.parser')
    soup.find('tr', class_="over_header").decompose()
    mvp_table = soup.find(id="mvp")
    mvp_df = pd.read_html(StringIO(str(mvp_table)))[0]
    mvp_df['Year'] = year
    
    dfs.append(mvp_df)

In [11]:
mvps = pd.concat(dfs)

In [12]:
mvps.tail()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,...,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48,Year
7,8,Kevin Durant,30,GSW,0,25,1010,0.025,78,34.6,...,6.4,5.9,0.7,1.1,0.521,0.353,0.885,11.5,0.204,2019
8,9,Kawhi Leonard,27,TOR,0,13,1010,0.013,60,34.0,...,7.3,3.3,1.8,0.4,0.496,0.371,0.854,9.5,0.224,2019
9,10,Russell Westbrook,30,OKC,0,8,1010,0.008,73,36.0,...,11.1,10.7,1.9,0.5,0.428,0.290,0.656,6.8,0.124,2019
10,11T,Rudy Gobert,26,UTA,0,1,1010,0.001,81,31.8,...,12.9,2.0,0.8,2.3,0.669,NaN,0.636,14.4,0.268,2019
11,11T,LeBron James,34,LAL,0,1,1010,0.001,55,35.2,...,8.5,8.3,1.3,0.6,0.510,0.339,0.665,7.2,0.179,2019


In [13]:
mvps.to_csv("mvps.csv")

In [14]:
player_stats_url = "https://www.basketball-reference.com/leagues/NBA_{}_per_game.html"

url = player_stats_url.format(1991)
data = requests.get(url)
with open("player_stats/1991.html", "w+") as f:
    f.write(data.text)

In [38]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

In [40]:
brave_path = '/snap/bin/brave'

options = Options()
options.binary_location = brave_path
options.add_argument('--remote-debugging-port=9224')

# Selenium will automatically match and manage the driver binary
driver = webdriver.Chrome(options=options)

In [42]:
for year in years:
    url = player_stats_url.format(year)
    
    driver.get(url)
    driver.execute_script("window.scrollTo(1,10000)")
    time.sleep(2)
    
    with open("player_stats/{}.html".format(year), "w+") as f:
        f.write(driver.page_source)

In [49]:
dfs = []
for year in years:
    with open("player_stats/{}.html".format(year)) as f:
        page = f.read()
    
    soup = BeautifulSoup(page, 'html.parser')
    soup.find('tr', class_="thead").decompose()
    player_table = soup.find_all(id="per_game_stats")[0]
    player_df = pd.read_html(StringIO(str(player_table)))[0]
    player_df["Year"] = year
    dfs.append(player_df)

In [50]:
players = pd.concat(dfs)

In [52]:
players.head()

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Year
0,1,Michael Jordan,27,CHI,SG,82,82,37.0,12.1,22.4,...,4.6,6.0,5.5,2.7,1.0,2.5,2.8,31.5,"MVP-1,DPOY-7,AS,NBA1,DEF1",1991
1,2,Karl Malone,27,UTA,PF,82,82,40.3,10.3,19.6,...,8.9,11.8,3.3,1.1,1.0,3.0,3.3,29.0,"MVP-5,AS,NBA1",1991
2,3,Bernard King,34,WSB,SF,64,64,37.5,11.1,23.6,...,3.2,5.0,4.6,0.9,0.3,4.0,2.9,28.4,"MVP-16,AS,NBA3",1991
3,4,Charles Barkley,27,PHI,SF,67,67,37.3,9.9,17.4,...,6.3,10.1,4.2,1.6,0.5,3.1,2.6,27.6,"MVP-4,AS,NBA1",1991
4,5,Patrick Ewing,28,NYK,C,81,81,38.3,10.4,20.3,...,8.8,11.2,3.0,1.0,3.2,3.6,3.5,26.6,"MVP-11,DPOY-7,AS,NBA2",1991


In [53]:
players.to_csv("players.csv")

In [54]:
team_stats_url = "https://www.basketball-reference.com/leagues/NBA_{}_standings.html"

In [56]:
for year in years:
    url = team_stats_url.format(year)
    
    data = requests.get(url)
    
    with open("team/{}.html".format(year), "w+") as f:
        f.write(data.text)

In [59]:
dfs = []
for year in years:
    with open("team/{}.html".format(year)) as f:
        page = f.read()
    
    soup = BeautifulSoup(page, 'html.parser')
    soup.find('tr', class_="thead").decompose()
    e_table = soup.find_all(id="divs_standings_E")[0]
    e_df = pd.read_html(StringIO(str(e_table)))[0]
    e_df["Year"] = year
    e_df["Team"] = e_df["Eastern Conference"]
    del e_df["Eastern Conference"]
    dfs.append(e_df)
    
    w_table = soup.find_all(id="divs_standings_W")[0]
    w_df = pd.read_html(StringIO(str(w_table)))[0]
    w_df["Year"] = year
    w_df["Team"] = w_df["Western Conference"]
    del w_df["Western Conference"]
    dfs.append(w_df)

In [60]:
teams = pd.concat(dfs)

In [61]:
teams.tail()

,W,L,W/L%,GB,PS/G,PA/G,SRS,Year,Team
13,53,29,.646,—,113.9,109.1,4.96,2019,Houston Rockets*
14,48,34,.585,5.0,111.7,110.0,1.80,2019,San Antonio Spurs*
15,33,49,.402,20.0,103.5,106.1,-2.08,2019,Memphis Grizzlies
16,33,49,.402,20.0,115.4,116.8,-1.10,2019,New Orleans Pelicans
17,33,49,.402,20.0,108.9,110.1,-0.87,2019,Dallas Mavericks


In [62]:
teams.to_csv("teams.csv")